# From an epoch loop to saved-model inference

This notebook revisits the single neuron from *Neural Networks from Scratch*. It learns $y = 2x + 1$ from five examples. You can inspect one training step, watch the loss fall, and reuse the trained model in a fresh session.

Run all cells on the first pass. On a later visit, run **Setup** and **Load and predict** only. Use the same working directory, or set `checkpoint_path` to the saved file's location.

The notebook uses PyTorch on the CPU with data defined in the cells. If needed, run `%pip install torch` in a new cell.

## Setup

`nn.Linear(1, 1)` implements $\hat y = wx+b$ and expects a batch with one feature per row. The same architecture is used when we reload the weights. The checkpoint path is relative to the kernel's working directory.

In [ ]:
from pathlib import Path

import torch
from torch import nn

checkpoint_path = Path("epoch_loop_linear_model.pt")

## Train and save

An **epoch** is one pass over the training examples. Here we put all five examples in one batch, so one epoch includes one optimizer update. With several batches, the same five operations happen for each batch inside the epoch.

We start the weight and bias at zero so the first update is easy to follow. The mean squared error is the average of $(\hat y-y)^2$ over the batch.

In [ ]:
train_inputs = torch.tensor([[-2.0], [-1.0], [0.0], [1.0], [2.0]])
train_targets = 2 * train_inputs + 1

model = nn.Linear(1, 1)
nn.init.zeros_(model.weight)
nn.init.zeros_(model.bias)

loss_fn = nn.MSELoss()
learning_rate = 0.1
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

### Inspect the five operations

1. **Zero the gradients.** PyTorch accumulates gradients in each parameter's `.grad`. Clear them so the next update uses the current batch.
2. **Forward pass.** Use the current weight and bias to compute a prediction for each input.
3. **Compute the loss.** Compare predictions with the targets and reduce the errors to one number.
4. **Backward pass.** Autograd applies the chain rule to compute the derivative of the loss with respect to each parameter.
5. **Update the parameters.** SGD subtracts the learning rate times the gradient from each parameter. A gradient of $-8$ with a learning rate of $0.1$ moves the weight from $0$ to $0.8$.

The printed loss is measured before each logged update. The fit is evaluated again after the final update.

In [ ]:
epochs = 60
model.train()

for epoch in range(1, epochs + 1):
    # 1. Zero the gradients.
    optimizer.zero_grad()

    # 2. Forward pass.
    predictions = model(train_inputs)

    # 3. Compute the loss.
    loss = loss_fn(predictions, train_targets)

    # 4. Backward pass.
    loss.backward()
    if epoch == 1:
        print("First forward pass:", predictions.detach().flatten().tolist())
        print(f"First gradients: dw={model.weight.grad.item():.3f}, db={model.bias.grad.item():.3f}")

    # 5. Update the parameters.
    optimizer.step()
    if epoch == 1:
        print(f"After one update: w={model.weight.item():.3f}, b={model.bias.item():.3f}")

    if epoch in (1, 5, 20, epochs):
        print(f"Epoch {epoch:2d} | loss before update {loss.item():.6f}")

### Save the learned state

A model's `state_dict()` maps names to parameter tensors and any persistent buffers. This model has two entries, `weight` and `bias`. Saving them is enough for inference when we know the model architecture. To resume training with an optimizer such as Adam, a training checkpoint would also typically store the optimizer state and epoch.

We record predictions for three unseen inputs so we can check the reload. Running this cell again writes the current weights to the same path. [PyTorch save and load tutorial](https://docs.pytorch.org/tutorials/beginner/basics/saveloadrun_tutorial.html)

In [ ]:
model.eval()
with torch.inference_mode():
    final_loss = loss_fn(model(train_inputs), train_targets).item()
    reference_inputs = torch.tensor([[-3.0], [0.5], [4.0]])
    reference_predictions = model(reference_inputs)

state = model.state_dict()
torch.save(state, checkpoint_path)

print(f"Final training loss: {final_loss:.2e}")
for name, tensor in state.items():
    print(f"{name}: {tensor.item():.6f}")
print(f"Saved {checkpoint_path}")

## Load and predict

After restarting the kernel, run **Setup** and the two code cells below. They use the checkpoint on disk and can run without the training data or optimizer.

Create a fresh `nn.Linear(1, 1)`, load its state onto the CPU, and switch it to evaluation mode. `eval()` controls the behavior of layers such as dropout and batch normalization. `torch.inference_mode()` disables gradient tracking during the forward pass. [PyTorch inference mode reference](https://docs.pytorch.org/docs/stable/generated/torch.autograd.grad_mode.inference_mode.html)

In [ ]:
loaded_model = nn.Linear(1, 1)
saved_state = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
loaded_model.load_state_dict(saved_state)
loaded_model.eval()
print(f"Loaded {checkpoint_path}")

In [ ]:
new_inputs = torch.tensor([[-3.0], [0.5], [4.0]])

with torch.inference_mode():
    loaded_predictions = loaded_model(new_inputs)

for x, prediction in zip(new_inputs.flatten().tolist(), loaded_predictions.flatten().tolist()):
    print(f"x={x:4.1f} | prediction {prediction:7.4f} | expected {2 * x + 1:5.1f}")

### Verify the round trip

When training and inference ran in the same session, compare the predictions from both models on the same inputs. Matching exactly shows that saving and loading preserved this model's behavior for those inputs.

In [ ]:
torch.testing.assert_close(new_inputs, reference_inputs, rtol=0, atol=0)
torch.testing.assert_close(loaded_predictions, reference_predictions, rtol=0, atol=0)
print("The predictions match exactly before and after loading.")